# 🤖 Module 6: Product Recommendation Engine

In this notebook, we build a **Cross-Selling Product Recommendation Engine** powered by association rules mined from customer transaction histories.

---

In [1]:
import os
import pandas as pd
import numpy as np

rules_path = os.path.join("..", "data", "processed", "association_rules.csv")
rules_df = pd.read_csv(rules_path)
print(f"Loaded {len(rules_df):,} association rules.")

Loaded 1,030 association rules.


### Step 1: Recommendation Function Definition

In [2]:
def get_top_recommendations(product_name: str, rules: pd.DataFrame, top_n: int = 5) -> list:
    """Query association rules for cross-selling product recommendations."""
    if product_name is None or product_name.strip() == "":
        return []
    
    query = product_name.strip().upper()
    matches = rules[rules['antecedents'].str.upper().str.contains(query, regex=False, na=False)]
    
    if not matches.empty:
        matches = matches.sort_values(['lift', 'confidence'], ascending=[False, False])
        recs = []
        for consequents in matches['consequents']:
            items = [item.strip() for item in consequents.split(',')]
            for item in items:
                if item.upper() != query and item not in recs:
                    recs.append(item)
                if len(recs) >= top_n:
                    break
            if len(recs) >= top_n:
                break
        return recs[:top_n]
    return []

### Step 2: Test Product Recommendation Engine

In [3]:
test_product = "WHITE HANGING HEART T-LIGHT HOLDER"
recommendations = get_top_recommendations(test_product, rules_df, top_n=5)

print(f"🛍️ Input Product: '{test_product}'")
print("🚀 Top 5 Co-Purchased Recommendations:")
for i, rec in enumerate(recommendations, 1):
    print(f"  {i}. {rec}")

🛍️ Input Product: 'WHITE HANGING HEART T-LIGHT HOLDER'
🚀 Top 5 Co-Purchased Recommendations:
  1. CANDLEHOLDER PINK HANGING HEART
  2. RED HANGING HEART T-LIGHT HOLDER
  3. HEART OF WICKER LARGE
  4. WOODEN PICTURE FRAME WHITE FINISH
  5. WOODEN FRAME ANTIQUE WHITE


### Step 3: Test Recommendation Engine on Multiple Products

In [4]:
sample_products = [
    "REGENCY CAKESTAND 3 TIER",
    "HEART OF WICKER SMALL",
    "JUMBO BAG RED RETROSPOT",
    "ROSES REGENCY TEACUP AND SAUCER"
]

for prod in sample_products:
    recs = get_top_recommendations(prod, rules_df, top_n=3)
    print(f"\nInput: {prod}")
    print("Recommendations:", recs)


Input: REGENCY CAKESTAND 3 TIER
Recommendations: ['ROSES REGENCY TEACUP AND SAUCER', 'PINK REGENCY TEACUP AND SAUCER', 'GREEN REGENCY TEACUP AND SAUCER']

Input: HEART OF WICKER SMALL
Recommendations: ['HEART OF WICKER LARGE', 'SMALL WHITE HEART OF WICKER', 'WHITE HANGING HEART T-LIGHT HOLDER']

Input: JUMBO BAG RED RETROSPOT
Recommendations: ['JUMBO BAG STRAWBERRY', 'JUMBO BAG PINK POLKADOT', 'JUMBO STORAGE BAG SUKI']

Input: ROSES REGENCY TEACUP AND SAUCER
Recommendations: ['REGENCY CAKESTAND 3 TIER', 'GREEN REGENCY TEACUP AND SAUCER', 'PINK REGENCY TEACUP AND SAUCER']
